# Matched discrete-time RWM factorial analysis

Adapted from the repository comparison template and the audited predecessor world-model notebook. Prospective scripts read W&B only, preserve all original decision reducers, and keep old diagnostics exploratory. D1 is the sole promotion candidate. No interval or population significance claim is made with two training seeds.


In [ ]:
GROUP = "rwm-discrete-flow"
PROJECT = "rwm-explore"
ENTITY = "oxdynamics"
BASE = "B0"
EVAL_RUN_IDS = None
NOTES = {"evidence": "Read the paired seed effects and criterion table before interpreting the mechanism.",
         "tradeoffs": "Consistency identities alone cannot establish prediction accuracy.",
         "confounders": "Population generalization remains inconclusive with the authorized replication. Shadow action sequences differ between candidates.",
         "candidates": "The active researcher decides after reviewing the criterion classifications and failure coverage.",
         "next": "No follow-up experiment is authorized by this comparison."}


In [ ]:
import json
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb
from IPython.display import Markdown, display
from research_workflow.metric_resolution import MetricResolver
from research_workflow.scientific_review import build_evidence, identifier

# Copy API values into local mappings; delegated capabilities are guarded reads only.
from types import SimpleNamespace

def copied_api_value(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, (list, tuple)):
        return [copied_api_value(item) for item in value]
    if callable(getattr(value, "items", None)) and callable(getattr(value, "keys", None)):
        return {key: copied_api_value(item) for key, item in value.items()}
    raise TypeError("Unsupported API value; refusing to omit or stringify evidence")

def run_values(run):
    values = {name: getattr(run, name) for name in
              ("id", "name", "job_type", "state", "url", "group", "commit")}
    values["summary"] = copied_api_value(run.summary)
    values["config"] = copied_api_value(run.config)
    values["tags"] = list(run.tags)
    # Match MetricResolver's public history-metadata observation, including unavailable None.
    values["history_keys"] = copied_api_value(getattr(run, "history_keys", None))
    for name in ("history", "logged_artifacts", "used_artifacts"):
        values[name] = getattr(run, name)
    return SimpleNamespace(**values)

api = wandb.Api()
entity = ENTITY or api.default_entity
all_runs = [run_values(run) for run in api.runs(f"{entity}/{PROJECT}", filters={"group": GROUP})]
train = [run for run in all_runs if run.job_type == "train"]
evals = [run for run in all_runs if run.job_type == "eval" and "status:superseded" not in run.tags]
if EVAL_RUN_IDS is not None:
    assert len(EVAL_RUN_IDS) == len(set(EVAL_RUN_IDS))
    evals = [run for run in evals if run.id in EVAL_RUN_IDS]
FIGURES, TABLES, PROBLEMS = {}, {}, []
EVIDENCE_ARTIFACTS = []
def keep_table(name, frame):
    TABLES[name] = frame.reset_index(drop=True)
    return frame

def keep_fig(name, figure):
    FIGURES[name] = figure
    plt.close(figure)
    return figure

def research(run):
    return (run.config.get("runner_cfg") or run.config.get("train_cfg") or {}).get("research") or run.config.get("research") or {}


# SDKs may expose dictionary summaries as dotted keys. Restore only their representation.
def summary_mapping(summary, name):
    raw = dict(summary)
    result = {}
    prefix = name + '.'
    for key, value in raw.items():
        if not key.startswith(prefix):
            continue
        parts = key[len(prefix):].split('.')
        target = result
        for part in parts[:-1]:
            target = target.setdefault(part, {})
            if not isinstance(target, dict):
                raise ValueError('Conflicting summary dictionary paths')
        if parts[-1] in target and target[parts[-1]] != value:
            raise ValueError('Conflicting summary values')
        target[parts[-1]] = value
    if name in raw:
        nested = dict(raw[name])
        if result and nested != result:
            raise ValueError('Nested and dotted summary representations disagree')
        return nested
    if not result:
        raise KeyError(name)
    return result


In [ ]:
expected_cells = {(variant, seed) for variant in ("B0", "B1", "B2", "D1", "E3_direct_velocity") for seed in (33, 44)}
assert {(research(run)["variant"], research(run)["seed"]) for run in evals} == expected_cells
assert len(evals) == len(expected_cells), "Ambiguous attempts require explicit reviewed EVAL_RUN_IDS"
training_by_id = {run.id: run for run in train}
frames, trajectories, stochastic_frames, inventory_rows, efficiency, cost_rows = [], [], [], [], [], []
protocols, fingerprints, campaign_costs = [], [], []
for run in sorted(evals, key=lambda run: run.id):
    rs = research(run)
    parent = training_by_id[rs["train_run_id"]]
    assert parent.state == run.state == "finished"
    prs = research(parent)
    assert all(rs[key] == prs[key] for key in ("variant", "seed", "protocol_id"))
    assert parent.config["world_model"]["iterations"] == 20000
    assert parent.config["world_model"]["batch_windows"] == 256
    artifacts = [artifact for artifact in run.logged_artifacts() if artifact.type == "world-model-evaluation"]
    assert len(artifacts) == 1, "Require one exact attached scored artifact"
    artifact = artifacts[0]
    assert artifact.metadata["checkpoint_sha256"] == rs["checkpoint_sha256"]
    assert artifact.metadata["protocol_id"] == rs["protocol_id"]
    scored = artifact.get("episodes").get_dataframe()
    assert len(scored) == 8 * 256 * 6
    keys = ["evaluation_partition", "mode", "checkpoint_role", "episode_id", "horizon_steps"]
    assert not scored.duplicated(keys).any()
    for _, block in scored.groupby(keys[:3]):
        assert set(zip(block.episode_id.astype(int), block.horizon_steps.astype(int))) == {(episode,horizon) for episode in range(256) for horizon in (10,50,100,250,500,1500)}
    assert scored.variant.eq(rs["variant"]).all() and scored.training_seed.eq(rs["seed"]).all()
    factors = rs["protocol_contract"]["evaluation"]["paired_factors"]
    for row in scored.to_dict("records"):
        assert json.loads(row["eval_unit_id"]) == {factor: row[factor] for factor in factors}
        if not row["initialization_valid"]:
            assert pd.isna(row["unstable"]) and pd.isna(row["stable_duration_s"])
    frames.append(scored)
    trajectories.append(artifact.get("trajectory_samples").get_dataframe())
    sampled = artifact.get("stochastic").get_dataframe()
    assert len(sampled) == 16 * 256 * 6
    assert set(sampled.sample_stream.astype(int)) == set(range(6100,6116))
    assert not sampled.duplicated(["sample_stream","episode_id","horizon_steps"]).any()
    stochastic_frames.append(sampled)
    EVIDENCE_ARTIFACTS.append({"eval_run_id":run.id,"name":artifact.qualified_name,"digest":artifact.digest})
    fp = prs["runtime_fingerprint"]
    assert not str(fp["rsl_rl_commit"]).endswith("-dirty")
    fingerprints.append(tuple(fp[key] for key in ("commit","rsl_rl_commit","isaaclab_commit")))
    protocols.append(rs["protocol_contract"])
    inventory_rows.append({"variant":rs["variant"],"train_seed":rs["seed"],"run_id":parent.id,"eval_run_id":run.id,
                           "checkpoint_sha256":rs["checkpoint_sha256"],"protocol_id":rs["protocol_id"],"source_commit":fp["commit"]})
    efficiency.append({"variant":rs["variant"],"train_seed":rs["seed"],**summary_mapping(run.summary,"timing")})
    campaign_costs.append(summary_mapping(run.summary,"campaign_cost"))
assert len(set(fingerprints)) == 1, "Executable source differs across trained models"
assert all(protocol == protocols[0] for protocol in protocols)
assert all(cost == campaign_costs[0] for cost in campaign_costs)
protocol = protocols[0]
assert protocol["group"] == GROUP and protocol["seeds"] == [33,44]
assert protocol["world_model"]["collection"]["test_id"]["seed"] == 7301
assert protocol["world_model"]["collection"]["test_id"]["command_seed"] == 8301
assert protocol["world_model"]["collection"]["test_command_shift"]["seed"] == 7302
assert protocol["world_model"]["collection"]["test_command_shift"]["command_seed"] == 8302
assert protocol["world_model"]["dataset_partition_reuse"]["artifact"] == "oxdynamics/rwm-explore/rwm-physical-consistency-dataset:v0"
assert len({research(run)["dataset_id"] for run in evals}) == 1
assert all(research(run)["dataset_id"] != "4272832c98da50d7e78c4dfc3fee9bbc4ca477c2e1bf6326194dbf8ca0d0e494" for run in evals)
episodes = pd.concat(frames, ignore_index=True)
trajectory_samples = pd.concat(trajectories, ignore_index=True)
stochastic = pd.concat(stochastic_frames, ignore_index=True)
keep_table("inventory",pd.DataFrame(inventory_rows))
keep_table("efficiency",pd.DataFrame(efficiency))
for run in train:
    rs=research(run)
    cost_rows.append({"run_id":run.id,"variant":rs.get("variant"),"train_seed":rs.get("seed"),"state":run.state,
                      "selected_attempt":run.id in set(TABLES["inventory"].run_id),
                      "loop_GPU_hours":run.summary.get("cost/GPU_hours"),"parameters":run.summary.get("parameters"),
                      "updates":run.summary.get("train/updates")})
keep_table("cost_attempts",pd.DataFrame(cost_rows))
keep_table("campaign_cost",pd.DataFrame([{"GPU":gpu,"allocated_hours":hours} for gpu,hours in campaign_costs[0]["allocated_GPU_hours_per_device"].items()]))
METRIC_RESOLVER=MetricResolver(all_runs)
display(TABLES["inventory"])


In [ ]:
group_keys = ["variant","training_seed","evaluation_partition","mode","horizon_steps","checkpoint_role"]
metric_columns = [column for column in episodes.columns if column.endswith(("rmse","_rad","_rad_s","_m","_m_s","_Nm","_fraction","_error","_F1","_Hz"))]
metric_columns = list(dict.fromkeys(metric_columns + ["physical_trajectory_nrmse","failure_aware_score","joint_position_rmse_rad","joint_velocity_rmse_rad_s","command_tracking_linear_rmse","command_tracking_yaw_rmse","unstable","numerical_divergence","stable_duration_s","reference_terminated_before_horizon","gravity_pose_defect"]))
summary_rows=[]
for keys, block in episodes.groupby(group_keys):
    row=dict(zip(group_keys,keys))
    row.update(train_seed=int(row["training_seed"]),slice=f"{row['evaluation_partition']}/{row['mode']}/{int(row['horizon_steps'])}/{row['checkpoint_role']}",
               planned=len(block),initialized=int(block.initialization_valid.sum()),reference_eligible=int(block.reference_eligible.sum()),
               finite_reference_pairs=int((block.reference_eligible & block.prediction_finite).sum()))
    for metric in metric_columns:
        values=pd.to_numeric(block[metric],errors="coerce").dropna()
        row[metric]=float(values.mean()) if len(values) else np.nan
        row[metric+"__n"]=len(values)
    physical=block.physical_trajectory_nrmse.dropna()
    row["physical_nrmse_p90"]=physical.quantile(.9) if len(physical) else np.nan
    row["physical_nrmse_p95"]=physical.quantile(.95) if len(physical) else np.nan
    alias=f"{row['evaluation_partition']}/{row['mode']}/{int(row['horizon_steps'])}/failure_aware_score"
    if row["checkpoint_role"]=="selected":row[alias]=row["failure_aware_score"]
    summary_rows.append(row)
per_seed=keep_table("per_seed",pd.DataFrame(summary_rows))
display(per_seed[(per_seed.evaluation_partition=="test_id") & (per_seed["mode"]=="A") & (per_seed.horizon_steps==100) & (per_seed.checkpoint_role=="selected")])


In [ ]:
contrasts=[("D1","B0"),("D1","B1"),("E3_direct_velocity","B2"),("B2","B1"),("E3_direct_velocity","D1"),("B1","B0"),("B2","B0"),("E3_direct_velocity","B0")]
join_keys=["training_seed","evaluation_partition","episode_id","fork_step","command_schedule_id","mode","horizon_steps","checkpoint_role"]
paired_rows=[]
for candidate,baseline in contrasts:
    merged=episodes[episodes.variant==candidate].merge(episodes[episodes.variant==baseline],on=join_keys,suffixes=("_candidate","_baseline"),validate="one_to_one")
    assert len(merged)==len(episodes[episodes.variant==candidate])
    for keys,block in merged.groupby(["training_seed","evaluation_partition","mode","horizon_steps","checkpoint_role"]):
        identity=dict(zip(["train_seed","partition","mode","horizon_steps","checkpoint_role"],keys))
        for metric in ("failure_aware_score","physical_trajectory_nrmse","joint_position_rmse_rad","joint_velocity_rmse_rad_s","command_tracking_linear_rmse","command_tracking_yaw_rmse","unstable","numerical_divergence"):
            x=pd.to_numeric(block[metric+"_candidate"],errors="coerce")
            y=pd.to_numeric(block[metric+"_baseline"],errors="coerce")
            matched=x.notna() & y.notna()
            if metric in ("physical_trajectory_nrmse","joint_position_rmse_rad","joint_velocity_rmse_rad_s"):
                matched = matched & block.reference_eligible_candidate & block.reference_eligible_baseline & block.prediction_finite_candidate & block.prediction_finite_baseline
            x,y=x[matched].astype(float),y[matched].astype(float)
            delta=x-y
            paired_rows.append({**identity,"variant":candidate,"baseline":baseline,"metric":metric,"planned":len(block),"paired_n":len(x),
                                "candidate_mean":x.mean(),"baseline_mean":y.mean(),"mean_delta":delta.mean(),"median_delta":delta.median(),
                                "relative_improvement":1-x.mean()/y.mean() if len(x) and y.mean()>1e-8 else np.nan,
                                "directional_fraction":(delta<0).mean() if len(x) else np.nan,
                                "action_interpretation":"different candidate actions, paired initial conditions" if identity["mode"] in ("B","B_shadow") else "identical recorded actions"})
paired=keep_table("world_model_paired_effects",pd.DataFrame(paired_rows))
effects=[]
for candidate in ("B1","B2","D1","E3_direct_velocity"):
    key="test_id/A/100/failure_aware_score"
    selected=per_seed[(per_seed.variant.isin([candidate,"B0"])) & per_seed["slice"].eq("test_id/A/100/selected")]
    pivot=selected.pivot(index="train_seed",columns="variant",values=key).dropna()
    delta=(pivot[candidate]-pivot["B0"]).tolist()
    effects.append({"variant":candidate,"baseline":"B0","metric":key,"slice":"test_id/A/100/selected","delta":np.mean(delta) if delta else np.nan,"seed_effects":delta})
keep_table("effects",pd.DataFrame(effects))
primary=paired[(paired.variant=="D1") & (paired.baseline=="B0") & paired.partition.eq("test_id") & paired["mode"].eq("A") & paired.horizon_steps.eq(100) & paired.checkpoint_role.eq("selected")]
display(primary)
print("Two independent training-seed pairs: no aggregate confidence interval or significance claim.")


In [ ]:
criteria=[]
def assess(name,values,rule,scope):
    values=np.asarray(values,dtype=float)
    finite=np.isfinite(values)
    classification="inconclusive" if not len(values) or not finite.all() else ("met" if rule(values) else "not_met")
    criteria.append({"criterion":name,"classification":classification,"scope":scope,"values":values[finite].tolist(),"valid_values":int(finite.sum()),"planned_values":len(values)})
main=paired[paired.variant.eq("D1") & paired.baseline.eq("B0") & paired.partition.eq("test_id") & paired.checkpoint_role.eq("selected")]
for metric in ("failure_aware_score","physical_trajectory_nrmse"):
    p=main[main.metric.eq(metric) & main["mode"].eq("A") & main.horizon_steps.eq(100)]
    assess("two_second_20_percent_"+metric,p.relative_improvement,lambda x:len(x)==2 and x.mean()>=.20 and (x>0).all(),"equal seed weight; common eligible pairs")
    p=main[main.metric.eq(metric) & main["mode"].eq("A") & main.horizon_steps.isin([50,100,250])]
    # Ratio of horizon-mean scores within each seed, then equal seed weight.
    ratios=[]
    for seed in (33,44):
        horizons=p[p.train_seed.eq(seed)]
        usable=(len(horizons)==3 and set(horizons.horizon_steps.astype(int))=={50,100,250}
                and (horizons.paired_n>0).all()
                and np.isfinite(horizons[["candidate_mean","baseline_mean"]].to_numpy()).all()
                and (horizons.baseline_mean>1e-8).all())
        ratios.append(1-horizons.candidate_mean.mean()/horizons.baseline_mean.mean() if usable else np.nan)
    ratio=np.asarray(ratios)
    assess("one_two_five_second_15_percent_"+metric,ratio,lambda x:len(x)==2 and x.mean()>=.15 and (x>0).all(),"equal horizon then seed weight")
    p=main[main.metric.eq(metric) & main["mode"].eq("B_shadow") & main.horizon_steps.eq(100)]
    assess("shadow_two_second_15_percent_"+metric,p.relative_improvement,lambda x:len(x)==2 and x.mean()>=.15 and (x>0).all(),"own actions; paired common reference support")
    p=main[main.metric.eq(metric) & main["mode"].eq("A") & main.horizon_steps.eq(10)]
    assess("short_horizon_max_5_percent_"+metric,-p.relative_improvement,lambda x:len(x)==2 and (x<=.05).all(),"each training seed")
for metric in ("joint_position_rmse_rad","joint_velocity_rmse_rad_s"):
    p=main[main.metric.eq(metric) & main["mode"].eq("A") & main.horizon_steps.isin([50,100,250])]
    assess(metric+"_max_5_percent",-p.relative_improvement,lambda x:len(x)==6 and (x<=.05).all(),"each seed at each declared accuracy horizon")
p=main[main.metric.eq("unstable") & main["mode"].eq("B") & main.horizon_steps.eq(1500)]
assess("autonomous_instability_max_2pp",p.mean_delta,lambda x:len(x)==2 and (x<=.02).all(),"initialized model population, reference-independent")
for metric in ("command_tracking_linear_rmse","command_tracking_yaw_rmse"):
    p=main[main.metric.eq(metric) & main["mode"].eq("B")]
    assess(metric+"_max_10_percent",-p.relative_improvement,lambda x:len(x)==12 and (x<=.10).all(),"each training seed and all declared horizons")
p=main[main.metric.eq("numerical_divergence") & main["mode"].isin(["A","B"])]
assess("no_increased_model_divergence",p.mean_delta,lambda x:len(x)==24 and (x<=0).all(),"each seed and horizon in A and B")
eff=TABLES["efficiency"]
lat=eff[eff.variant.eq("D1")].merge(eff[eff.variant.eq("B0")],on="train_seed",suffixes=("_candidate","_baseline"),validate="one_to_one")
same_latency_interface=lat.device_candidate.eq(lat.device_baseline) & lat.dtype_candidate.eq(lat.dtype_baseline) & lat.model_latency_p95_ms_baseline.gt(1e-8)
assess("model_p95_max_20_percent",np.where(same_latency_interface,lat.model_latency_p95_ms_candidate/lat.model_latency_p95_ms_baseline-1,np.nan),lambda x:len(x)==2 and (x<=.2).all(),"same GPU model/dtype, each seed")
assess("policy_plus_model_p95_below_20ms",np.where(same_latency_interface,lat.policy_plus_model_latency_p95_ms_candidate,np.nan),lambda x:len(x)==2 and (x<20).all(),"each seed")
criteria.append({"criterion":"population_generalization","classification":"inconclusive","scope":"two training seeds; no population-support claim","values":[],"valid_values":2,"planned_values":2})
keep_table("acceptance_criteria",pd.DataFrame(criteria))
display(TABLES["acceptance_criteria"])


In [ ]:
# Full factorial effects use common support across all four cells, per training seed.
factorial_variants = ["B1","B2","D1","E3_direct_velocity"]
keys = ["training_seed","evaluation_partition","episode_id","fork_step","command_schedule_id","mode","horizon_steps","checkpoint_role"]
columns = ["reference_eligible","prediction_finite","initialization_valid","failure_aware_score","physical_trajectory_nrmse",
           "joint_position_rmse_rad","joint_velocity_rmse_rad_s","translation_rmse_m","orientation_rmse_rad",
           "command_tracking_linear_rmse","command_tracking_yaw_rmse","unstable","numerical_divergence"]
wide = None
for variant in factorial_variants:
    frame = episodes[episodes.variant.eq(variant) & episodes.checkpoint_role.eq("selected")][keys+columns].copy()
    assert not frame.duplicated(keys).any()
    frame = frame.rename(columns={column:column+"__"+variant for column in columns})
    wide = frame if wide is None else wide.merge(frame,on=keys,validate="one_to_one")
assert len(wide) == len(episodes[episodes.variant.eq("B1") & episodes.checkpoint_role.eq("selected")])
factorial_rows=[]
for group,block in wide.groupby(["training_seed","evaluation_partition","mode","horizon_steps","checkpoint_role"]):
    identity=dict(zip(["train_seed","partition","mode","horizon_steps","checkpoint_role"],group))
    for metric in columns[3:]:
        mask=np.ones(len(block),dtype=bool)
        for variant in factorial_variants:
            mask &= pd.to_numeric(block[metric+"__"+variant],errors="coerce").notna().to_numpy()
            if metric in ["failure_aware_score","physical_trajectory_nrmse","joint_position_rmse_rad",
                          "joint_velocity_rmse_rad_s","translation_rmse_m","orientation_rmse_rad"]:
                mask &= block["reference_eligible__"+variant].astype(bool).to_numpy()
                if metric != "failure_aware_score":
                    mask &= block["prediction_finite__"+variant].astype(bool).to_numpy()
            else:
                mask &= block["initialization_valid__"+variant].astype(bool).to_numpy()
        means={variant:pd.to_numeric(block.loc[mask,metric+"__"+variant],errors="coerce").astype(float).mean()
               for variant in factorial_variants}
        direct=((means["D1"]-means["B1"])+(means["E3_direct_velocity"]-means["B2"]))/2
        integrated=((means["B2"]-means["B1"])+(means["E3_direct_velocity"]-means["D1"]))/2
        interaction=(means["E3_direct_velocity"]-means["D1"])-(means["B2"]-means["B1"])
        factorial_rows.append({**identity,"metric":metric,"planned":len(block),"common_n":int(mask.sum()),
            **{variant+"_mean":value for variant,value in means.items()},"direct_minus_residual":direct,
            "integrated_minus_independent":integrated,"interaction":interaction,
            "interpretation":"identical recorded actions" if identity["mode"]=="A" else "own-action outcomes or own-action shadow accuracy; references/actions differ"})
factorial=keep_table("factorial_effects",pd.DataFrame(factorial_rows))
figure,axes=plt.subplots(1,3,figsize=(13,4),sharey=True)
selected=factorial[factorial.partition.eq("test_id") & factorial["mode"].eq("A") &
                   factorial.horizon_steps.eq(100) & factorial.metric.eq("failure_aware_score")]
for axis,effect in zip(axes,["direct_minus_residual","integrated_minus_independent","interaction"]):
    axis.bar(selected.train_seed.astype(str),selected[effect])
    axis.axhline(0,color="black",linewidth=.6)
    axis.set(title=effect,xlabel="independent training seed")
axes[0].set_ylabel("Signed two-second cap-score difference; lower is better")
keep_fig("factorial-main-effects-and-interaction",figure)
display(selected)


In [ ]:
selected=per_seed[per_seed.checkpoint_role.eq("selected") & per_seed.evaluation_partition.eq("test_id")]
for mode in ("A","B","B_shadow"):
    fig,ax=plt.subplots(figsize=(8,4))
    for (variant,seed),block in selected[selected["mode"].eq(mode)].groupby(["variant","train_seed"]):
        block=block.sort_values("horizon_steps")
        ax.plot(block.horizon_steps*.02,block.failure_aware_score,marker="o",label=f"{variant} seed {seed}")
    ax.set(xscale="log",xlabel="horizon (s)",ylabel="failure-aware physical trajectory score",title=mode+": separate evaluation semantics")
    ax.legend(fontsize=6,ncol=2)
    keep_fig("physical-error-"+mode.lower(),fig)
for metric in ("joint_integration_defect_rad","gravity_pose_defect","gravity_norm_error","joint_limit_violation_fraction"):
    fig,ax=plt.subplots(figsize=(7,3))
    if metric not in selected:continue
    for variant,block in selected[selected["mode"].eq("A")].groupby("variant"):
        values=block.groupby("horizon_steps")[metric].mean()
        ax.plot(values.index*.02,values.values,marker="o",label=variant)
    ax.set(xscale="log",xlabel="horizon (s)",ylabel=metric)
    ax.legend(fontsize=7)
    keep_fig("consistency-"+metric.replace("_","-"),fig)
for mode in ("A","B"):
    block=trajectory_samples[(trajectory_samples.episode_id==0) & (trajectory_samples.training_seed==33) & trajectory_samples["mode"].eq(mode)]
    fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True)
    for variant,frame in block.groupby("variant"):
        valid=frame.reference_valid.astype(bool) & frame.prediction_finite.astype(bool) & frame.initialization_valid.astype(bool)
        distance=np.sqrt(sum((frame["predicted_com_"+axis]-frame["actual_com_"+axis])**2 for axis in "xyz"))
        axes[0].plot(frame.time_s,np.where(valid,distance,np.nan),label=variant)
        axes[1].plot(frame.time_s,np.where(valid,frame.orientation_error_rad,np.nan),label=variant)
    axes[0].set(ylabel="COM translation error (m)")
    axes[1].set(xlabel="time (s)",ylabel="orientation error (rad)")
    axes[0].legend(fontsize=7)
    keep_fig("trajectory-"+mode.lower(),fig)
    position_columns=[column for column in block if column.startswith("predicted_position_")]
    for kind in ("position","velocity"):
        columns=[column for column in block if column.startswith("predicted_"+kind+"_")]
        fig,axes=plt.subplots(4,3,figsize=(13,10),sharex=True)
        for ax,column in zip(axes.flat,columns):
            for variant,frame in block.groupby("variant"):
                valid=frame.reference_valid.astype(bool) & frame.prediction_finite.astype(bool) & frame.initialization_valid.astype(bool)
                ax.plot(frame.time_s,np.where(valid,frame[column],np.nan),label=variant,linewidth=.6)
            actual=block[block.variant.eq("B0")]
            ax.plot(actual.time_s,np.where(actual.reference_valid.astype(bool),actual[column.replace("predicted_","actual_",1)],np.nan),color="black",linewidth=.8,label="actual")
            ax.set_title(column.removeprefix("predicted_"),fontsize=8)
        axes.flat[0].legend(fontsize=6)
        keep_fig("joints-"+kind+"-"+mode.lower(),fig)


In [ ]:
# Additional predeclared descriptive slices, all from immutable W&B Tables.
ood = episodes[episodes.evaluation_partition.eq("test_command_shift")].copy()
ood["command_stratum"] = (ood.episode_id.astype(int) // 64).map(
    {0: "fast_forward", 1: "lateral", 2: "turning", 3: "reversals"}
)
strata = []
for keys, block in ood.groupby(
    ["variant", "training_seed", "mode", "horizon_steps", "checkpoint_role", "command_stratum"]
):
    row = dict(zip(["variant", "training_seed", "mode", "horizon_steps", "checkpoint_role", "command_stratum"], keys))
    row.update(
        planned=len(block),
        initialized=int(block.initialization_valid.sum()),
        eligible=int(block.reference_eligible.sum()),
        finite_pairs=int((block.reference_eligible & block.prediction_finite).sum()),
    )
    for metric in (
        "physical_trajectory_nrmse",
        "failure_aware_score",
        "command_tracking_linear_rmse",
        "command_tracking_yaw_rmse",
        "unstable",
    ):
        values = pd.to_numeric(block[metric], errors="coerce").dropna()
        row[metric] = float(values.mean()) if len(values) else np.nan
        row[metric + "__n"] = len(values)
    strata.append(row)
keep_table("command_shift_strata", pd.DataFrame(strata))
stream_scores = []
for keys, block in stochastic.groupby(["variant", "training_seed", "sample_stream", "horizon_steps"]):
    stream_scores.append(
        dict(zip(["variant", "training_seed", "sample_stream", "horizon_steps"], keys))
        | {
            "score": pd.to_numeric(block.failure_aware_score, errors="coerce").mean(),
            "eligible": int(block.reference_eligible.sum()),
            "finite_reference_pairs": int((block.reference_eligible & block.prediction_finite).sum()),
        }
    )
stream_scores = pd.DataFrame(stream_scores)
keep_table("stochastic_stream_diagnostics", stream_scores)
keep_table(
    "stochastic_stream_spread",
    stream_scores.groupby(["variant", "training_seed", "horizon_steps"])
    .agg(
        mean_score=("score", "mean"),
        stream_sd=("score", "std"),
        streams=("score", "count"),
        eligible=("eligible", "min"),
    )
    .reset_index(),
)
# This spread is within one trained model; it is not a confidence interval over training seeds.
final = per_seed[per_seed.checkpoint_role.eq("final") & per_seed["mode"].eq("A")]
selected = per_seed[per_seed.checkpoint_role.eq("selected") & per_seed["mode"].eq("A")]
keys = ["variant", "train_seed", "evaluation_partition", "horizon_steps"]
sensitivity = final.merge(selected, on=keys, suffixes=("_final", "_selected"), validate="one_to_one")
sensitivity["cap_score_final_minus_selected"] = (
    sensitivity.failure_aware_score_final - sensitivity.failure_aware_score_selected
)
sensitivity["finite_score_final_minus_selected"] = (
    sensitivity.physical_trajectory_nrmse_final - sensitivity.physical_trajectory_nrmse_selected
)
keep_table(
    "final_checkpoint_sensitivity",
    sensitivity[
        keys
        + [
            "cap_score_final_minus_selected",
            "finite_score_final_minus_selected",
            "reference_eligible_final",
            "reference_eligible_selected",
        ]
    ],
)
# The publisher reports only the active pool. First-pool queue allocations are
# bound in the approved resource amendment and its immutable ledger hashes.
resource_amendment = protocol["world_model"]["resource_amendment"]
carried_device_hours = resource_amendment["carried_current_comparison_GPU_hours_per_device"]
carried_phase_hours = resource_amendment["carried_current_comparison_GPU_hours_per_phase"]
original_prior_device_hours = resource_amendment["original_physical_prior_device_GPU_hours"]
carried_cost = sum(carried_device_hours.values())
original_prior_cost = resource_amendment["original_physical_comparison_GPU_hours"]
assert not (set(carried_device_hours) & set(campaign_costs[0]["allocated_GPU_hours_per_device"]))
full_phase_hours = dict(campaign_costs[0]["allocated_GPU_hours_per_phase"])
for phase, hours in carried_phase_hours.items():
    full_phase_hours[phase] = full_phase_hours.get(phase, 0) + hours
known_phase = sum(full_phase_hours.values())
active_cost = sum(campaign_costs[0]["allocated_GPU_hours_per_device"].values())
current_campaign_cost = carried_cost + active_cost
prior_device_hours = protocol["world_model"]["device_placement"]["prior_device_hours"]
prior_cost = sum(prior_device_hours.values())
assert abs(prior_cost - original_prior_cost - carried_cost) < 1e-8
total_cost = active_cost + prior_cost
assert abs(total_cost - original_prior_cost - current_campaign_cost) < 1e-8
keep_table("resource_transition_carry", pd.DataFrame([
    {"GPU": gpu, "carried_current_comparison_hours": hours,
     "source": "closed original-pool queue ledgers bound in approved protocol"}
    for gpu, hours in carried_device_hours.items()
]))
keep_table("all_current_comparison_phase_cost", pd.DataFrame([
    {"phase": phase, "GPU_hours": hours} for phase, hours in full_phase_hours.items()
]))
keep_table("campaign_cost_reconciliation", pd.DataFrame([{
    "total_allocated_GPU_hours": total_cost, "listed_new_phase_GPU_hours": known_phase,
    "prior_comparison_GPU_hours": original_prior_cost, "new_campaign_GPU_hours": current_campaign_cost,
    "closed_first_pool_GPU_hours": carried_cost, "active_pool_GPU_hours": active_cost,
    "new_unlisted_GPU_hours": current_campaign_cost-known_phase,
    "source": "prior immutable audited allocation plus new conservative queue ledger, each counted once"}]))
complete_device_cost = {gpu: prior_device_hours.get(gpu,0) + campaign_costs[0]["allocated_GPU_hours_per_device"].get(gpu,0)
                        for gpu in sorted(set(prior_device_hours) | set(campaign_costs[0]["allocated_GPU_hours_per_device"]))}
keep_table("campaign_cost",pd.DataFrame([{"GPU":gpu,"allocated_hours":hours,
    "original_prior_hours":original_prior_device_hours.get(gpu,0),
    "closed_first_pool_hours":carried_device_hours.get(gpu,0),
    "new_hours":carried_device_hours.get(gpu,0)+campaign_costs[0]["allocated_GPU_hours_per_device"].get(gpu,0)} for gpu,hours in complete_device_cost.items()]))
assert sum(complete_device_cost.values()) <= 24.0 + 1e-8
assert all(value <= 12.0 + 1e-8 for value in complete_device_cost.values())
assert current_campaign_cost <= 14.2652631025 + 1e-8

# Complete the predeclared validity/contact-proxy curves. Numerical validity and
# representational identities remain diagnostics, never evidence of accuracy.
validation_metrics = [
    ("rotation_norm_error", "quaternion norm defect"),
    ("rotation_orthogonality_error", "rotation orthogonality defect"),
    ("rotation_determinant_error", "rotation determinant defect"),
    ("joint_limit_excess_rad", "maximum joint-limit excess (rad)"),
    ("nominal_plane_foot_origin_penetration_m", "nominal foot-origin plane penetration (m; geometry proxy)"),
    ("predicted_support_foot_speed_m_s", "predicted-support foot speed (m/s; geometry proxy)"),
]
validity = per_seed[
    per_seed.checkpoint_role.eq("selected") & per_seed.evaluation_partition.eq("test_id") & per_seed["mode"].eq("A")
]
fig, axes = plt.subplots(3, 2, figsize=(13, 11), sharex=True)
for ax, (metric, label) in zip(axes.flat, validation_metrics):
    if metric not in validity:
        ax.text(0.5, 0.5, "unavailable", ha="center", transform=ax.transAxes)
        ax.set_title(label)
        continue
    for (variant, seed), block in validity.groupby(["variant", "train_seed"]):
        block = block.sort_values("horizon_steps")
        ax.plot(
            block.horizon_steps * 0.02,
            block[metric],
            marker="o",
            markersize=3,
            linestyle="-" if seed == 33 else "--",
            label=f"{variant} seed {seed}",
        )
    ax.set(xscale="log", xlabel="horizon (s)", ylabel=label)
axes.flat[0].legend(fontsize=6, ncol=2)
keep_fig("rotation-and-contact-proxy-validity", fig)
behaviour_metrics = [
    ("gait_frequency_error_Hz", "gait-frequency discrepancy (Hz)"),
    ("per_leg_phase_error_rad", "per-leg phase discrepancy (rad; no time warp)"),
    ("joint_amplitude_error_rad", "joint-amplitude discrepancy (rad)"),
    ("command_tracking_linear_rmse", "linear command-tracking RMSE (m/s)"),
    ("command_tracking_yaw_rmse", "yaw command-tracking RMSE (rad/s)"),
    ("unstable", "initialized-rollout instability fraction"),
]
behaviour = per_seed[
    per_seed.checkpoint_role.eq("selected") & per_seed.evaluation_partition.eq("test_id") & per_seed["mode"].eq("B")
]
fig, axes = plt.subplots(3, 2, figsize=(13, 11), sharex=True)
for ax, (metric, label) in zip(axes.flat, behaviour_metrics):
    if metric not in behaviour:
        ax.text(0.5, 0.5, "unavailable", ha="center", transform=ax.transAxes)
        continue
    for (variant, seed), block in behaviour.groupby(["variant", "train_seed"]):
        block = block.sort_values("horizon_steps")
        ax.plot(
            block.horizon_steps * 0.02,
            block[metric],
            marker="o",
            markersize=3,
            linestyle="-" if seed == 33 else "--",
            label=f"{variant} seed {seed}",
        )
    ax.set(xscale="log", xlabel="horizon (s)", ylabel=label)
axes.flat[0].legend(fontsize=6, ncol=2)
fig.suptitle("Frozen-policy autonomous behaviour; trajectories use different actions")
keep_fig("autonomous-behaviour-diagnostics", fig)

physical_metrics = [
    ("translation_rmse_m", "COM translation RMSE (m)"),
    ("orientation_rmse_rad", "orientation geodesic RMSE (rad)"),
    ("linear_velocity_rmse_m_s", "body linear velocity RMSE (m/s)"),
    ("angular_velocity_rmse_rad_s", "body angular velocity RMSE (rad/s)"),
    ("joint_position_rmse_rad", "joint position RMSE (rad)"),
    ("joint_velocity_rmse_rad_s", "joint velocity RMSE (rad/s)"),
]
fig, axes = plt.subplots(3, 2, figsize=(13, 11), sharex=True)
for ax, (metric, label) in zip(axes.flat, physical_metrics):
    for (variant, seed), block in validity.groupby(["variant", "train_seed"]):
        block = block.sort_values("horizon_steps")
        ax.plot(
            block.horizon_steps * 0.02,
            block[metric],
            marker="o",
            markersize=3,
            linestyle="-" if seed == 33 else "--",
            label=f"{variant} seed {seed}",
        )
    ax.set(xscale="log", xlabel="horizon (s)", ylabel=label)
axes.flat[0].legend(fontsize=6, ncol=2)
fig.suptitle("Recorded-action physical prediction; finite eligible pairs, coverage reported separately")
keep_fig("physical-state-errors-by-horizon", fig)


In [ ]:
NUMBER=re.compile(r"\d+(\.\d+)?\s?%|\b\d+\s?/\s?\d+\b|\b\d+\.\d+\b")
assert not any(NUMBER.search(value) for value in NOTES.values())
primary_summary=per_seed[per_seed.evaluation_partition.eq("test_id") & per_seed["mode"].eq("A") & per_seed.horizon_steps.eq(100) & per_seed.checkpoint_role.eq("selected")]
lines=["### Comparison", "Recorded-action physical fidelity, autonomous outcome discrepancy and candidate-action shadows are distinct estimands.",
       "| Variant | Seed | Two-second cap score | Finite-pair score | Eligible | Initialized |", "|---|---:|---:|---:|---:|---:|"]
for row in primary_summary.itertuples():
    lines.append(f"| {row.variant} | {int(row.train_seed)} | {row.failure_aware_score:.5g} | {row.physical_trajectory_nrmse:.5g} | {row.reference_eligible}/{row.planned} | {row.initialized}/{row.planned} |")
lines += ["### Seed effects and criteria", "Raw seed effects and common-support denominators are retained in the world_model_paired_effects table."]
for row in TABLES["acceptance_criteria"].itertuples():
    lines.append(f"- {row.criterion}: {row.classification}; measured values {row.values}; valid {row.valid_values}/{row.planned_values}.")
lines += ["### Trade-offs",NOTES["tradeoffs"],"### Confounders / limitations",NOTES["confounders"],
          "Finite-pair metrics are conditional on reference survival and numerical validity. The capped score retains model failures. Initial failures remain population failures, never zero errors.",
          "Nominal foot-link geometry and gravity diagnostics do not prove collision-surface nonpenetration or force balance.",
          "Videos are illustrative, immutable-array kinematic replays; no replacement episodes enter scored evidence.",
          "### Candidate(s) worth considering",NOTES["candidates"],"### Suggested next check",NOTES["next"]]
DRAFT="\n".join(lines)
display(Markdown(DRAFT))
print("<<<ANALYSIS_DRAFT\n"+DRAFT+"\nANALYSIS_DRAFT>>>")


In [ ]:
# Deterministic evidence precedes any generated scientific interpretation.
from research_workflow.scientific_review import build_evidence

SCIENTIFIC_EVIDENCE = build_evidence(
    all_runs, project=PROJECT, group=GROUP, entity=entity,
    tables=TABLES, problems=PROBLEMS, baseline=BASE, resolver=METRIC_RESOLVER,
    hypothesis=json.loads(os.environ.get("RESEARCH_HYPOTHESIS", "null")),
)
SCIENTIFIC_EVIDENCE["analysis_code"] = json.loads(os.environ.get("RESEARCH_ANALYSIS_CODE", "null"))
from research_workflow.scientific_review import identifier
SCIENTIFIC_EVIDENCE["evidence_id"] = identifier("evidence", SCIENTIFIC_EVIDENCE)
print("<<<SCIENTIFIC_EVIDENCE\n" + json.dumps(SCIENTIFIC_EVIDENCE, allow_nan=False) + "\nSCIENTIFIC_EVIDENCE>>>")


In [ ]:
if os.environ.get("RESEARCH_ANALYSIS_LOG") == "1":
    source = json.loads(os.environ.get("RESEARCH_SOURCE", "{}"))
    run = wandb.init(project=PROJECT, entity=entity, group=GROUP, job_type="analysis", name=f"{GROUP}__analysis",
                     save_code=True, tags=["dirty"] if source.get("dirty") else [],
                     config={"research": {"comparison": GROUP, "notebook": f"research/{GROUP}/analysis.ipynb",
                                          "source": source, "baseline": BASE, "train_runs": [r.name for r in train],
                                          "eval_runs": [r.name for r in evals], "evidence_artifacts": EVIDENCE_ARTIFACTS}})
    kept = {f"analysis/fig/{slug}": wandb.Image(fig) for slug, fig in FIGURES.items()}
    kept |= {f"analysis/table/{slug}": wandb.Table(dataframe=df) for slug, df in TABLES.items()}
    run.log(kept)
    run.summary["analysis/scientific_evidence"] = SCIENTIFIC_EVIDENCE
    run.summary["analysis/draft_md"] = DRAFT
    run.summary["analysis/problems"] = len(PROBLEMS)
    run.finish()
    print("logged", sorted(kept))
else:
    print("interactive run: nothing written to W&B; publish with scripts/research_runs.py run-analysis")